Single data source (exactly as requested)

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("/Users/trungnguyenvan/Documents/SGH/3rd semester/Thesis/master data")

# Show files in folder (helps you pick the correct dataset file)
files = sorted([p.name for p in DATA_PATH.iterdir() if p.is_file()])
print("Files in DATA_PATH:")
for f in files:
    print("-", f)

Files in DATA_PATH:
- .DS_Store
- All_data.xlsx
- Marketplace_Data.csv
- Marketplace_Data.xlsx


In [10]:
DATA_FILE = DATA_PATH / "Marketplace_Data.csv"  # <-- replace with the correct filename
df_raw = pd.read_csv(DATA_FILE)

print("Loaded:", DATA_FILE.name)
print("Shape:", df_raw.shape)
df_raw.head(3)

Loaded: Marketplace_Data.csv
Shape: (73448, 24)


,venue_id,venue_name,room_id,room_name,venue_description,room_description,city,country,venue_types,venue_part,min_attendees_per_event,minimum_booking_hours,selected_for_online_payments,pricing_type,space_venue_type_label,room_min_price,room_max_price,pricing_methods,successful_enquiry_count,enquiry_count,number_of_verified_reviews,total_score,average_score,room_created_at
0,18984,Bond Collective East Austin,39786,Bouldin,Our first location in Texas is housed within t...,This completely private meeting room seats up ...,Austin,US,"[""meeting-centre"",""coworking-space""]",NaN,NaN,2.000,0,simplified,NaN,5000,5000,[hire fee],0,0,NaN,NaN,NaN,2022-07-27 14:28:36 UTC
1,18982,Charlie's,39785,Charlie’s,Charlie’s by the Charlie Borrow workshop. \n\n...,NaN,London,GB,"[""pub-bar""]",NaN,NaN,NaN,0,custom,NaN,15000,15000,[hire fee minimum spend],0,6,NaN,NaN,NaN,2022-07-27 14:22:04 UTC
2,21037,Abel's on the Lake,43883,Banqueting Hall,Abel’s on the Lake is the prime spot to come e...,It's time to plan your big event with Abel's o...,Austin,US,"[""restaurant""]",NaN,NaN,NaN,0,custom,NaN,2000,2000,[per person],0,2,NaN,NaN,NaN,2022-10-17 10:04:39 UTC


In [11]:
df_raw.columns.tolist()


['venue_id',
 'venue_name',
 'room_id',
 'room_name',
 'venue_description',
 'room_description',
 'city',
 'country',
 'venue_types',
 'venue_part',
 'min_attendees_per_event',
 'minimum_booking_hours',
 'selected_for_online_payments',
 'pricing_type',
 'space_venue_type_label',
 'room_min_price',
 'room_max_price',
 'pricing_methods',
 'successful_enquiry_count',
 'enquiry_count',
 'number_of_verified_reviews',
 'total_score',
 'average_score',
 'room_created_at']

1️⃣ Data integrity (keys, duplicates, unit of analysis)

In [6]:
# Check uniqueness of room_id
dup_room = df.duplicated(subset=["room_id"]).sum()
print("Duplicate room_id rows:", dup_room)

# Rooms per venue
rooms_per_venue = df.groupby("venue_id")["room_id"].nunique().describe()
rooms_per_venue


Duplicate room_id rows: 23664


count   27,132.000
mean         1.835
std          1.553
min          1.000
25%          1.000
50%          1.000
75%          2.000
max         30.000
Name: room_id, dtype: float64

2️⃣ Target definition & distribution (CRITICAL)

In [7]:
df["successful_enquiry_count"] = (
    pd.to_numeric(df["successful_enquiry_count"], errors="coerce")
    .fillna(0)
    .astype(int)
)

df["has_success"] = (df["successful_enquiry_count"] > 0).astype(int)

df["has_success"].value_counts(normalize=True).rename("share")


has_success
0   0.763
1   0.237
Name: share, dtype: float64

In [15]:
df["successful_enquiry_count"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)


count   73,448.000
mean         1.957
std         12.868
min          0.000
50%          0.000
75%          0.000
90%          3.000
95%          8.000
99%         37.000
max      1,232.000
Name: successful_enquiry_count, dtype: float64

In [16]:
zero_rate = (df["successful_enquiry_count"] == 0).mean()
print("Zero success rate:", round(zero_rate, 3))


Zero success rate: 0.763


3️⃣ Missingness analysis (feature selection gate)

In [17]:
missing_pct = (
    df.isna().mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
)

missing_pct.reset_index().rename(
    columns={"index": "column", 0: "missing_pct"}
)


,column,missing_pct
0,average_score,75.680
1,total_score,75.680
2,number_of_verified_reviews,75.680
3,minimum_booking_hours,69.620
4,min_attendees_per_event,52.010
5,space_venue_type_label,25.610
6,venue_part,22.860
7,venue_types,3.130
8,room_description,0.010
9,venue_description,0.000


In [18]:
missing_pct[missing_pct > 60]


average_score                75.680
total_score                  75.680
number_of_verified_reviews   75.680
minimum_booking_hours        69.620
dtype: float64

4️⃣ Numeric feature sanity checks

In [19]:
num_cols = [
    "room_min_price",
    "room_max_price",
    "min_attendees_per_event",
    "minimum_booking_hours",
    "number_of_verified_reviews",
    "total_score",
    "average_score"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df[num_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
room_min_price,"73,448.000","162,766.859","7,953,312.513",100.000,"5,500.000","15,000.000","100,000.000","2,147,483,647.000"
room_max_price,"73,448.000","240,152.784","7,967,786.171",100.000,"8,000.000","42,050.000","200,000.000","2,147,483,647.000"
min_attendees_per_event,"35,248.000",17.385,48.400,0.000,1.000,2.000,20.000,"2,500.000"
minimum_booking_hours,"22,312.000",2.616,1.917,1.000,1.000,2.000,4.000,12.000
number_of_verified_reviews,"17,861.000",6.957,14.496,1.000,1.000,3.000,7.000,236.000
total_score,"17,861.000",4.754,0.344,1.000,4.667,4.850,5.000,5.000
average_score,"17,861.000",2.378,1.836,0.020,0.711,1.667,4.750,5.000


In [20]:
# Logical price check
invalid_price = (
    df["room_min_price"].notna() &
    df["room_max_price"].notna() &
    (df["room_min_price"] > df["room_max_price"])
).sum()

print("Min price > max price rows:", invalid_price)


Min price > max price rows: 0


5️⃣ Categorical feature inspection

In [21]:
num_cols = [
    "room_min_price",
    "room_max_price",
    "min_attendees_per_event",
    "minimum_booking_hours",
    "number_of_verified_reviews",
    "total_score",
    "average_score"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df[num_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
room_min_price,"73,448.000","162,766.859","7,953,312.513",100.000,"5,500.000","15,000.000","100,000.000","2,147,483,647.000"
room_max_price,"73,448.000","240,152.784","7,967,786.171",100.000,"8,000.000","42,050.000","200,000.000","2,147,483,647.000"
min_attendees_per_event,"35,248.000",17.385,48.400,0.000,1.000,2.000,20.000,"2,500.000"
minimum_booking_hours,"22,312.000",2.616,1.917,1.000,1.000,2.000,4.000,12.000
number_of_verified_reviews,"17,861.000",6.957,14.496,1.000,1.000,3.000,7.000,236.000
total_score,"17,861.000",4.754,0.344,1.000,4.667,4.850,5.000,5.000
average_score,"17,861.000",2.378,1.836,0.020,0.711,1.667,4.750,5.000


In [22]:
# Logical price check
invalid_price = (
    df["room_min_price"].notna() &
    df["room_max_price"].notna() &
    (df["room_min_price"] > df["room_max_price"])
).sum()

print("Min price > max price rows:", invalid_price)


Min price > max price rows: 0
